[![Homepage](https://img.shields.io/badge/homepage-blueviolet?logo=htmx)](https://www.bendai.org/CUHK-STAT3009/)
[![GitHub](https://img.shields.io/badge/GitHub-black.svg?logo=github)](https://github.com/statmlben/CUHK-STAT3009)
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/statmlben/CUHK-STAT3009/blob/main/nb/STAT3009_baseline_methods.ipynb)

# Notebook 02 — Baseline methods for rating prediction

**STAT3009 · Recommender Systems**  
**Suggested time:** 75–90 minutes

This notebook follows one complete recommender-system workflow:

```text
raw triples → encode IDs → NumPy arrays → learn from train
            → predict test ratings → evaluate RMSE → rank movies
```

By the end, you should be able to:

1. convert raw user and movie IDs into contiguous integer indices;
2. construct `X_train`, `y_train`, and `X_test`;
3. implement global-, user-, and item-mean baselines with NumPy;
4. combine the three means with a simple arithmetic average;
5. turn predicted ratings into a recommendation ranking.

## 1. Setup

We use pandas to read the course data, NumPy for every baseline method, and `LabelEncoder` only for categorical ID encoding.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

pd.set_option('display.max_rows', 12)
pd.set_option('display.float_format', lambda x: f'{x:.3f}')

## 2. Load the Netflix course split

The public files contain `(user_id, movie_id, rating)` triples. We intentionally read both IDs as strings because they are **categorical labels**, not numerical measurements.

In [ ]:
BASE_URL = (
    'https://raw.githubusercontent.com/'
    'statmlben/CUHK-STAT3009/main/dataset/netflix'
)

TRAIN_URL = f'{BASE_URL}/train.csv'
TEST_URL = f'{BASE_URL}/test.csv'
COLUMNS = ['user_id', 'movie_id', 'rating']
ID_TYPES = {'user_id': 'string', 'movie_id': 'string'}

train_raw = pd.read_csv(
    TRAIN_URL, usecols=COLUMNS, dtype=ID_TYPES
)[COLUMNS]

test_raw = pd.read_csv(
    TEST_URL, usecols=COLUMNS, dtype=ID_TYPES
)[COLUMNS]

In [ ]:
print('train shape:', train_raw.shape)
print('test shape: ', test_raw.shape)
display(train_raw.head())
display(train_raw.dtypes)

The first row is one observed interaction:

$$
(u,i,r)=(\text{user }1960,\text{ movie }670,4).
$$

It means that user `1960` gave movie `670` a rating of `4`. The numerical appearance of an ID carries no distance or ordering information.

### Hide the test ratings during prediction

The public test file includes ratings so that we can evaluate in class. A prediction method receives only the test `(user, movie)` pairs. We reveal `y_test` only when calculating RMSE.

In [ ]:
test_pairs_raw = test_raw[['user_id', 'movie_id']].copy()
y_test = test_raw['rating'].to_numpy(dtype=float)

display(test_pairs_raw.head())
print('hidden test-rating vector:', y_test.shape)

## 3. Data preprocessing: categorical IDs

Raw IDs may be strings, non-consecutive integers, or a mixture of both. NumPy lookup vectors are simplest when every category has a contiguous position:

```text
raw user IDs   {"u_17", "u_205", "u_9"}  →  {0, 1, 2}
raw movie IDs  {"m_A",  "m_Q",   "m_B"}  →  {0, 1, 2}
```

Users and movies require separate encoders because they are different categorical vocabularies.

### Why fit on train and test IDs together?

Some test users or movies never appear in training. A train-only encoder would fail when transforming those IDs. We therefore build one shared ID vocabulary from the train and test **ID columns**.

> This does not use test ratings. It only decides which array position represents each categorical ID.

In [ ]:
all_ids = pd.concat(
    [
        train_raw[['user_id', 'movie_id']],
        test_raw[['user_id', 'movie_id']],
    ],
    ignore_index=True,
)

user_encoder = LabelEncoder().fit(all_ids['user_id'])
item_encoder = LabelEncoder().fit(all_ids['movie_id'])

n_users = len(user_encoder.classes_)
n_items = len(item_encoder.classes_)

print('number of users:', n_users)
print('number of movies:', n_items)

In [ ]:
train = train_raw.copy()
test = test_raw.copy()

train['user_id'] = user_encoder.transform(train_raw['user_id'])
test['user_id'] = user_encoder.transform(test_raw['user_id'])

train['movie_id'] = item_encoder.transform(train_raw['movie_id'])
test['movie_id'] = item_encoder.transform(test_raw['movie_id'])

display(train.head())

In [ ]:
user_mapping = pd.DataFrame({
    'raw_user_id': user_encoder.classes_[:8],
    'encoded_user_id': np.arange(8),
})

item_mapping = pd.DataFrame({
    'raw_movie_id': item_encoder.classes_[:8],
    'encoded_movie_id': np.arange(8),
})

display(user_mapping)
display(item_mapping)

### Checkpoint

1. Why do we encode user IDs and movie IDs separately?
2. Why do we concatenate train and test IDs before fitting the encoders?
3. Why does using the test ID columns here not leak the test ratings?

## 4. The NumPy data contract

From this point onward, every method uses the same arrays:

| Array | Shape | Meaning |
|---|---:|---|
| `X_train` | `(n_train, 2)` | encoded training `(user, movie)` pairs |
| `y_train` | `(n_train,)` | observed training ratings |
| `X_test` | `(n_test, 2)` | encoded test `(user, movie)` pairs |
| `y_test` | `(n_test,)` | hidden ratings used only for evaluation |
| `y_pred` | `(n_test,)` | one predicted rating for each test pair |

In [ ]:
X_train = train[['user_id', 'movie_id']].to_numpy(dtype=int)
y_train = train['rating'].to_numpy(dtype=float)

X_test = test[['user_id', 'movie_id']].to_numpy(dtype=int)

print('X_train:', X_train.shape, X_train.dtype)
print('y_train:', y_train.shape, y_train.dtype)
print('X_test: ', X_test.shape, X_test.dtype)
print('y_test:  ', y_test.shape, y_test.dtype)

In [ ]:
assert X_train.shape[1] == 2
assert len(X_train) == len(y_train)
assert len(X_test) == len(y_test)
assert X_train[:, 0].min() >= 0
assert X_train[:, 1].min() >= 0
assert X_test[:, 0].max() < n_users
assert X_test[:, 1].max() < n_items

print('All array checks passed.')

## 5. What kind of prediction problem is this?

Before choosing a method, inspect the number of observations, the rating scale, sparsity, and whether test users or movies appear in training.

In [ ]:
n_train = len(y_train)
n_test = len(y_test)
density = n_train / (n_users * n_items)
pair_overlap = len(
    set(map(tuple, X_train)) & set(map(tuple, X_test))
)

print(f'training triples: {n_train:,}')
print(f'test pairs:       {n_test:,}')
print(f'users:            {n_users:,}')
print(f'movies:           {n_items:,}')
print(f'observed density: {density:.3%}')
print(f'train/test pair overlap: {pair_overlap:,}')

In [ ]:
rating_distribution = (
    pd.Series(y_train, name='rating')
    .value_counts()
    .sort_index()
    .rename('count')
    .to_frame()
)
rating_distribution['proportion'] = (
    rating_distribution['count'] / n_train
)

display(rating_distribution)
print('global training mean:', y_train.mean())

### Four types of test pair

A test pair may contain a user, a movie, or both that never appeared in training. This determines when a baseline needs a fallback.

In [ ]:
train_users = np.unique(X_train[:, 0])
train_items = np.unique(X_train[:, 1])

known_user = np.isin(X_test[:, 0], train_users)
known_item = np.isin(X_test[:, 1], train_items)

PAIR_TYPES = [
    'known user, known movie',
    'known user, new movie',
    'new user, known movie',
    'new user, new movie',
]

pair_type = np.select(
    [
        known_user & known_item,
        known_user & ~known_item,
        ~known_user & known_item,
    ],
    PAIR_TYPES[:3],
    default=PAIR_TYPES[3],
)

pair_type_counts = (
    pd.Series(pair_type, name='test_pair_type')
    .value_counts()
    .reindex(PAIR_TYPES, fill_value=0)
)
display(pair_type_counts.to_frame('rows'))

## 6. Evaluation with RMSE

Every method must produce a vector `y_pred` with one value per test row. We then calculate

$$
\operatorname{RMSE}(y,\hat y)=
\sqrt{\frac{1}{n}\sum_{j=1}^{n}(y_j-\hat y_j)^2}.
$$

Lower RMSE means that the predicted ratings are closer to the held-out ratings on average.

In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

## 7. Global mean baseline

The simplest complete predictor uses one number for every test pair:

$$
\mu=\frac{1}{|\mathcal R_{\mathrm{train}}|}
\sum_{(u,i,r)\in\mathcal R_{\mathrm{train}}}r,
\qquad \hat r_{ui}=\mu.
$$

In [ ]:
global_mean = np.mean(y_train)
y_pred_global = np.full(n_test, global_mean)
rmse_global = rmse(y_test, y_pred_global)

print(f'global mean: {global_mean:.3f}')
print(f'test RMSE:  {rmse_global:.3f}')

The global mean is both a baseline and a useful fallback. It can score every test pair, including a completely new user–movie pair.

## 8. User mean baseline

For each observed user, average that user's training ratings:

$$
\bar r_u=\frac{1}{|\mathcal I_u^{\mathrm{tr}}|}
\sum_{i\in\mathcal I_u^{\mathrm{tr}}}r_{ui},
\qquad \hat r_{ui}=\bar r_u.
$$

We first fill the vector with `global_mean`. The loop overwrites only the users observed in training, so test-only users automatically keep the fallback.

In [ ]:
user_mean = np.full(n_users, global_mean)

for u in np.unique(X_train[:, 0]):
    user_mean[u] = np.mean(y_train[X_train[:, 0] == u])

y_pred_user = user_mean[X_test[:, 0]]
rmse_user = rmse(y_test, y_pred_user)

print(f'test RMSE: {rmse_user:.3f}')

### Read the Boolean indexing expression

```python
np.mean(y_train[X_train[:, 0] == u])
```

Read it from the inside out:

1. `X_train[:, 0]` selects the user column;
2. `X_train[:, 0] == u` creates a Boolean mask;
3. `y_train[mask]` selects that user's ratings;
4. `np.mean(...)` averages the selected ratings.

In [ ]:
example_user_raw = '1960'
u = int(user_encoder.transform([example_user_raw])[0])
user_mask = X_train[:, 0] == u

print('raw user ID:       ', example_user_raw)
print('encoded user ID:   ', u)
print('number of ratings: ', user_mask.sum())
print('user mean:         ', np.mean(y_train[user_mask]))
print('stored value:      ', user_mean[u])

## 9. Item mean baseline

The item baseline repeats the same pattern on the movie column:

$$
\bar r_i=\frac{1}{|\mathcal U_i^{\mathrm{tr}}|}
\sum_{u\in\mathcal U_i^{\mathrm{tr}}}r_{ui},
\qquad \hat r_{ui}=\bar r_i.
$$

The only structural change is `X_train[:, 0]` → `X_train[:, 1]`.

### Your turn

Complete the item-mean implementation before opening the solution cell. Use the global mean as the initial value for every movie.

In [ ]:
# item_mean = ...
#
# for i in ...:
#     item_mean[i] = ...
#
# y_pred_item = ...

### Solution

In [ ]:
item_mean = np.full(n_items, global_mean)

for i in np.unique(X_train[:, 1]):
    item_mean[i] = np.mean(y_train[X_train[:, 1] == i])

y_pred_item = item_mean[X_test[:, 1]]
rmse_item = rmse(y_test, y_pred_item)

print(f'test RMSE: {rmse_item:.3f}')

## 10. Compare the three basic baselines

Each model uses a different amount of information. The comparison is specific to this train/test split.

In [ ]:
basic_results = pd.DataFrame({
    'method': ['global mean', 'user mean', 'item mean'],
    'test_RMSE': [rmse_global, rmse_user, rmse_item],
    'information': [
        'overall rating level',
        'user rating tendency',
        'movie rating tendency',
    ],
})

display(basic_results.sort_values('test_RMSE'))

### Verify the fallback behavior

Because both lookup vectors started at `global_mean`, an unseen user or movie already has a valid prediction.

In [ ]:
new_user = ~known_user
new_item = ~known_item
unseen_user_ids = np.setdiff1d(np.unique(X_test[:, 0]), train_users)
unseen_item_ids = np.setdiff1d(np.unique(X_test[:, 1]), train_items)

assert np.allclose(y_pred_user[new_user], global_mean)
assert np.allclose(y_pred_item[new_item], global_mean)

print('unique test-only users:', len(unseen_user_ids))
print('test rows with an unseen user:', new_user.sum())
print('unique test-only movies:', len(unseen_item_ids))
print('test rows with an unseen movie:', new_item.sum())
print('Fallback checks passed.')

## 11. Three-mean baseline

User mean and item mean capture different average tendencies. A simple first combination takes the arithmetic average of three already-computed numbers:

$$
\hat r_{ui}=\frac{\bar r_u+\bar r_i+\mu}{3}.
$$

This is intentionally simple. We are not fitting residuals and we are not tuning hyperparameters in this lecture.

In [ ]:
u_test = X_test[:, 0]
i_test = X_test[:, 1]

user_term = user_mean[u_test]
item_term = item_mean[i_test]

y_pred_three = (user_term + item_term + global_mean) / 3
rmse_three = rmse(y_test, y_pred_three)

print(f'test RMSE: {rmse_three:.3f}')

### One prediction, three numbers

The next cell reproduces the lecture example for user `1960` and movie `2098`.

In [ ]:
example_item_raw = '2098'
i = int(item_encoder.transform([example_item_raw])[0])

example_prediction = (user_mean[u] + item_mean[i] + global_mean) / 3

print(f'user mean:   {user_mean[u]:.3f}')
print(f'item mean:   {item_mean[i]:.3f}')
print(f'global mean: {global_mean:.3f}')
print(f'prediction:  {example_prediction:.3f}')

In [ ]:
all_results = pd.DataFrame({
    'method': [
        'global mean',
        'item mean',
        'user mean',
        'three-mean average',
    ],
    'test_RMSE': [
        rmse_global,
        rmse_item,
        rmse_user,
        rmse_three,
    ],
})

all_results['reduction_from_global'] = (
    rmse_global - all_results['test_RMSE']
)

display(all_results.sort_values('test_RMSE'))

### Performance by test-pair type

A single overall RMSE can hide cold-start behavior. We therefore inspect the same three-mean predictions in the four test-pair groups.

In [ ]:
segment_rows = []

for label in PAIR_TYPES:
    mask = pair_type == label
    segment_rows.append({
        'test_pair_type': label,
        'rows': int(mask.sum()),
        'three_mean_RMSE': rmse(y_test[mask], y_pred_three[mask]),
    })

segment_results = pd.DataFrame(segment_rows)
display(segment_results)

Do not over-interpret a segment with very few rows. In particular, a tiny full cold-start group cannot provide a reliable estimate of future performance.

## 12. Rating prediction returns to ranking

The model first creates one score for every test pair. For a particular user, sorting candidate movies by these scores produces a recommendation list.

```text
test pairs for one user → predicted ratings → descending sort → top-k movies
```

The course focuses on rating prediction, but the scores still have an engineering role inside a ranking pipeline.

In [ ]:
ranking_user_raw = example_user_raw
ranking_user = int(user_encoder.transform([ranking_user_raw])[0])

candidate_rows = np.flatnonzero(X_test[:, 0] == ranking_user)
ranking_order = np.argsort(-y_pred_three[candidate_rows])
top_rows = candidate_rows[ranking_order[:10]]

top_10 = pd.DataFrame({
    'rank': np.arange(1, len(top_rows) + 1),
    'user_id': ranking_user_raw,
    'movie_id': item_encoder.inverse_transform(X_test[top_rows, 1]),
    'predicted_rating': y_pred_three[top_rows],
})

print(f'user {ranking_user_raw}: {len(candidate_rows)} candidate movies')
display(top_10)

### Why can the three-mean model rank movies?

For one fixed user, the user mean and global mean stay constant. The item mean changes across candidate movies, so the final predicted ratings can be sorted.

A user-mean-only model would give every movie the same score for that user and therefore could not produce a meaningful movie ordering without another signal.

## 13. Flexible weights: a conceptual extension

The equal average is one member of a larger family:

$$
\hat r_{ui}=w_u\bar r_u+w_i\bar r_i+w_0\mu,
\qquad w_u+w_i+w_0=1.
$$

The next cell illustrates different fixed weights. It does **not** search for the best weights. Weight selection belongs to validation and model selection later in the course.

In [ ]:
w_user = 0.5
w_item = 0.3
w_global = 0.2

assert np.isclose(w_user + w_item + w_global, 1.0)

y_pred_weighted = (
    w_user * user_term
    + w_item * item_term
    + w_global * global_mean
)

display(pd.DataFrame({
    'equal_weight_prediction': y_pred_three[:5],
    'weighted_prediction': y_pred_weighted[:5],
}))

## Exit ticket

Answer these questions without running more code:

1. What does `user_mean[X_test[:, 0]]` return?
2. Why does initializing `user_mean` with `global_mean` solve the new-user problem?
3. Which term allows the three-mean model to rank different movies for one fixed user?
4. Why should we not choose weights by repeatedly checking `y_test`?

## Summary

| Method | Prediction for `(u, i)` | Fallback |
|---|---|---|
| Global mean | $\mu$ | always available |
| User mean | $\bar r_u$ | $\mu$ for a new user |
| Item mean | $\bar r_i$ | $\mu$ for a new movie |
| Three-mean average | $(\bar r_u+\bar r_i+\mu)/3$ | missing group means become $\mu$ |

The important reusable pattern is:

```text
learn lookup values from training triples
→ retrieve one value for every test pair
→ form a complete prediction vector
→ evaluate ratings or sort scores into a ranking
```